In [1]:
import pandas as pd
import re

# Simulate Apache access logs with attacks
sample_logs = [
    '192.168.1.10 - - [31/Jan/2026:10:00:01 +0000] "GET /index.php?id=1 HTTP/1.1" 200 1234 "-" "Mozilla/5.0"',
    '192.168.1.20 - - [31/Jan/2026:10:00:02 +0000] "GET /login HTTP/1.1" 403 345 "-" "Mozilla/5.0"',
    '192.168.1.20 - - [31/Jan/2026:10:00:03 +0000] "GET /login HTTP/1.1" 403 345 "-" "Mozilla/5.0"',
    '192.168.1.20 - - [31/Jan/2026:10:00:04 +0000] "GET /login HTTP/1.1" 403 345 "-" "Mozilla/5.0"',
    '192.168.1.30 - - [31/Jan/2026:10:00:05 +0000] "GET /search.php?q=1+UNION+SELECT+1,2 HTTP/1.1" 200 567 "-" "Mozilla/5.0"',
    '192.168.1.30 - - [31/Jan/2026:10:00:06 +0000] "GET /login.php?id=1\' OR \'1\'=\'1 HTTP/1.1" 200 890 "-" "bot"'
]

print(f"Loaded {len(sample_logs)} log lines")

Loaded 6 log lines


In [2]:
log_pattern = re.compile(
    r'(?P<ip>\S+) \S+ \S+ \[(?P<time>[^\]]+)\] '
    r'"(?P<method>\S+) (?P<path>\S+) (?P<proto>\S+)" '
    r'(?P<status>\d{3}) (?P<size>\S+) '
    r'"(?P<referrer>[^"]*)" "(?P<ua>[^"]*)"'
)

def parse_logs(log_lines):
    parsed = []
    for line in log_lines:
        match = log_pattern.match(line)
        if match:
            data = match.groupdict()
            data['status'] = int(data['status'])
            parsed.append(data)
    return pd.DataFrame(parsed)

df_logs = parse_logs(sample_logs)
df_logs.head()

,ip,time,method,path,proto,status,size,referrer,ua
0,192.168.1.10,31/Jan/2026:10:00:01 +0000,GET,/index.php?id=1,HTTP/1.1,200,1234,-,Mozilla/5.0
1,192.168.1.20,31/Jan/2026:10:00:02 +0000,GET,/login,HTTP/1.1,403,345,-,Mozilla/5.0
2,192.168.1.20,31/Jan/2026:10:00:03 +0000,GET,/login,HTTP/1.1,403,345,-,Mozilla/5.0
3,192.168.1.20,31/Jan/2026:10:00:04 +0000,GET,/login,HTTP/1.1,403,345,-,Mozilla/5.0
4,192.168.1.30,31/Jan/2026:10:00:05 +0000,GET,"/search.php?q=1+UNION+SELECT+1,2",HTTP/1.1,200,567,-,Mozilla/5.0


In [3]:
SQLI_PATTERNS = [
    r"UNION\s+SELECT",
    r"\'\s*OR\s*\'1\'=\'1",
    r"\'--",
    r"\"--",
    r"DROP\s+TABLE"
]

compiled_sqli = [re.compile(p, re.IGNORECASE) for p in SQLI_PATTERNS]

def detect_sqli(path):
    for pattern in compiled_sqli:
        if pattern.search(path):
            return True
    return False

df_logs['sqli_suspicious'] = df_logs['path'].apply(detect_sqli)
print("SQL Injection Results:")
print(df_logs[df_logs['sqli_suspicious']][['ip', 'path', 'status']])

SQL Injection Results:
Empty DataFrame
Columns: [ip, path, status]
Index: []


In [4]:
df_logs['time'] = pd.to_datetime(df_logs['time'], format='%d/%b/%Y:%H:%M:%S %z')
df_logs['time_bucket'] = df_logs['time'].dt.floor('1min')

df_403 = df_logs[df_logs['status'] == 403].copy()
brute_counts = df_403.groupby(['time_bucket', 'ip']).size().reset_index(name='count_403')
suspicious_brute = brute_counts[brute_counts['count_403'] >= 3]

print("Brute Force Results:")
print(suspicious_brute)

Brute Force Results:
                time_bucket            ip  count_403
0 2026-01-31 10:00:00+00:00  192.168.1.20          3


In [6]:
def siem_report():
    print("SIEM LOG ANALYSIS REPORT")
    print("=" * 40)
    
    sqli_attacks = df_logs[df_logs['sqli_suspicious']]
    if not sqli_attacks.empty:
        print("SQL INJECTION DETECTED:")
        for _, row in sqli_attacks.iterrows():
            print(f"  {row['ip']} -> {row['path']}")
    
    if not suspicious_brute.empty:
        print("\nBRUTE FORCE DETECTED:")
        for _, row in suspicious_brute.iterrows():
            print(f"  {row['ip']} -> {row['count_403']} failed logins/min")
    
    print("\nANALYSIS COMPLETE")

siem_report()

SIEM LOG ANALYSIS REPORT

BRUTE FORCE DETECTED:
  192.168.1.20 -> 3 failed logins/min

ANALYSIS COMPLETE
